# PySpark practice 12: Python UDFs, pandas and Arrow

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-12-python-udfs-and-pandas.ipynb)

12 exercises on Python and pandas UDFs, pandas function APIs, UDTFs, Arrow and replacing UDFs with built-ins. Each exercise names the [reference entries](pyspark-reference-12-python-udfs-and-pandas.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 12](pyspark-reference-12-python-udfs-and-pandas.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

from typing import Iterator

import pandas as pd

## Exercises

4 easy, 7 medium, 1 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Validate card numbers with a UDF](#exercise-1-validate-card-numbers-with-a-udf) | Medium | [`F.udf(f=None, returnType=StringType(), useArrow=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudffnone-returntypestringtype-usearrownone) |
| 2 | [Spot the silent null from a wrong return type](#exercise-2-spot-the-silent-null-from-a-wrong-return-type) | Medium | [`F.udf(f=None, returnType=StringType(), useArrow=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudffnone-returntypestringtype-usearrownone) |
| 3 | [Use a registered UDF from SQL](#exercise-3-use-a-registered-udf-from-sql) | Easy | [`spark.udf.register(name, f, returnType=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#sparkudfregistername-f-returntypenone) |
| 4 | [Vectorize with a pandas UDF](#exercise-4-vectorize-with-a-pandas-udf) | Easy | [`F.pandas_udf(f=None, returnType=None)`: Series to Series](pyspark-reference-12-python-udfs-and-pandas.ipynb#fpandas_udffnone-returntypenone-series-to-series) |
| 5 | [Custom aggregate: interquartile range per category](#exercise-5-custom-aggregate-interquartile-range-per-category) | Medium | [Grouped aggregate pandas UDFs](pyspark-reference-12-python-udfs-and-pandas.ipynb#grouped-aggregate-pandas-udfs) |
| 6 | [Per-group normalization with applyInPandas](#exercise-6-per-group-normalization-with-applyinpandas) | Medium | [`groupBy(...).applyInPandas(func, schema)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#groupbyapplyinpandasfunc-schema) |
| 7 | [Filter and reshape batches with mapInPandas](#exercise-7-filter-and-reshape-batches-with-mapinpandas) | Medium | [`mapInPandas(func, schema)` and `mapInArrow`](pyspark-reference-12-python-udfs-and-pandas.ipynb#mapinpandasfunc-schema-and-mapinarrow) |
| 8 | [As-of merge per customer with cogroup](#exercise-8-as-of-merge-per-customer-with-cogroup) | Hard | [`groupBy(...).cogroup(other.groupBy(...)).applyInPandas(func, schema)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#groupbycogroupothergroupbyapplyinpandasfunc-schema) |
| 9 | [Split text into rows with a Python UDTF](#exercise-9-split-text-into-rows-with-a-python-udtf) | Medium | [`@F.udtf(returnType=...)`: table functions](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudtfreturntype-table-functions) |
| 10 | [Replace a UDF with built-in functions](#exercise-10-replace-a-udf-with-built-in-functions) | Medium | [Cost and alternatives](pyspark-reference-12-python-udfs-and-pandas.ipynb#cost-and-alternatives) |
| 11 | [Find Python evaluation in a plan](#exercise-11-find-python-evaluation-in-a-plan) | Easy | [Arrow-optimized Python UDFs](pyspark-reference-12-python-udfs-and-pandas.ipynb#arrow-optimized-python-udfs) |
| 12 | [Round-trip through pandas with Arrow](#exercise-12-round-trip-through-pandas-with-arrow) | Easy | [`toPandas`, `createDataFrame(pandas_df)` and Arrow](pyspark-reference-12-python-udfs-and-pandas.ipynb#topandas-createdataframepandas_df-and-arrow) |

## Exercise 1: Validate card numbers with a UDF

**Difficulty:** Medium · about 10 min · **Practises:** [`F.udf(f=None, returnType=StringType(), useArrow=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudffnone-returntypestringtype-usearrownone)

*Some checks (like the Luhn checksum) have no built-in function.*

Write a UDF `luhn_ok(number)` returning true when the digits pass the Luhn check (null input gives null). Return `id` and `valid`, ordered by `id`.

In [ ]:
cards = spark.createDataFrame([(1, "4242424242424242"), (2, "4242424242424241"), (3, None), (4, "79927398713")], "id int, number string")
cards.show()

**Expected output**

| id | valid |
| --- | --- |
| 1 | true |
| 2 | false |
| 3 | null |
| 4 | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def luhn(number):
    if number is None:
        return None
    digits = [int(d) for d in number][::-1]
    total = sum(d if i % 2 == 0 else (d * 2 - 9 if d * 2 > 9 else d * 2) for i, d in enumerate(digits))
    return total % 10 == 0

luhn_ok = F.udf(luhn, "boolean")
result = cards.select("id", luhn_ok("number").alias("valid")).orderBy("id")
result.show()

## Exercise 2: Spot the silent null from a wrong return type

**Difficulty:** Medium · about 10 min · **Practises:** [`F.udf(f=None, returnType=StringType(), useArrow=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudffnone-returntypestringtype-usearrownone)

*A UDF declared with the wrong return type silently produces nulls; teams lose days to this.*

Apply `doubled_wrong` (declared `int`, returns a float) and a corrected UDF declared `double` to `qty`. Return `qty`, `wrong`, `fixed`, ordered by `qty`.

In [ ]:
nums = spark.createDataFrame([(1,), (2,), (3,)], "qty int")
doubled_wrong = F.udf(lambda q: q * 2.0, "int")

**Expected output**

| qty | wrong | fixed |
| --- | --- | --- |
| 1 | null | 2.0 |
| 2 | null | 4.0 |
| 3 | null | 6.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
doubled_fixed = F.udf(lambda q: q * 2.0, "double")
result = nums.select("qty", doubled_wrong("qty").alias("wrong"), doubled_fixed("qty").alias("fixed")).orderBy("qty")
result.show()

## Exercise 3: Use a registered UDF from SQL

**Difficulty:** Easy · about 5 min · **Practises:** [`spark.udf.register(name, f, returnType=None)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#sparkudfregistername-f-returntypenone)

*SQL users need access to Python logic too.*

Register a UDF `initials(name)` returning the uppercase initials of each word, then use it in SQL to return `customer_id` and `initials` for customers 1 to 3, ordered.

In [ ]:
customers.select("customer_id", "name").show(3)

**Expected output**

| customer_id | initials |
| --- | --- |
| 1 | AC |
| 2 | BR |
| 3 | CT |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.udf.register("initials", lambda name: "".join(w[0].upper() for w in name.split()) if name else None, "string")
result = spark.sql("SELECT customer_id, initials(name) AS initials FROM {c} WHERE customer_id <= 3 ORDER BY customer_id", c=customers)
result.show()

## Exercise 4: Vectorize with a pandas UDF

**Difficulty:** Easy · about 5 min · **Practises:** [`F.pandas_udf(f=None, returnType=None)`: Series to Series](pyspark-reference-12-python-udfs-and-pandas.ipynb#fpandas_udffnone-returntypenone-series-to-series)

*pandas UDFs process batches; the same logic runs far faster than a row UDF.*

Write a pandas UDF `gross(amount)` returning amount × 1.2 as double (rounded to 2 inside pandas). Return `order_id`, `gross` for orders 1 to 3, ordered.

In [ ]:
orders.select("order_id", "amount").show(3)

**Expected output**

| order_id | gross |
| --- | --- |
| 1 | 144.0 |
| 2 | 101.4 |
| 3 | 36.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def gross_values(amount: pd.Series) -> pd.Series:
    return (amount.astype(float) * 1.2).round(2)

gross = F.pandas_udf(gross_values, "double")
result = orders.where("order_id <= 3").select("order_id", gross("amount").alias("gross")).orderBy("order_id")
result.show()

## Exercise 5: Custom aggregate: interquartile range per category

**Difficulty:** Medium · about 15 min · **Practises:** [Grouped aggregate pandas UDFs](pyspark-reference-12-python-udfs-and-pandas.ipynb#grouped-aggregate-pandas-udfs)

*Robust spread statistics need custom aggregates.*

Write a grouped aggregate pandas UDF returning the interquartile range (75th minus 25th percentile, pandas default interpolation) of line values (qty × unit_price) per `category`. Return `category` and `iqr` rounded to 2, ordered by `category`.

In [ ]:
order_items.join(products, "sku").select("category", "qty", "unit_price").show(20)

**Expected output**

| category | iqr |
| --- | --- |
| Garden | 43.28 |
| Kitchen | 44.0 |
| Outdoor | 44.5 |
| Pantry | 282.69 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def iqr_value(values: pd.Series) -> float:
    v = values.astype(float)
    return float(v.quantile(0.75) - v.quantile(0.25))

iqr = F.pandas_udf(iqr_value, "double")
lines = order_items.join(products, "sku").withColumn("line", F.col("qty") * F.col("unit_price"))
result = lines.groupBy("category").agg(F.round(iqr("line"), 2).alias("iqr")).orderBy("category")
result.show()

## Exercise 6: Per-group normalization with applyInPandas

**Difficulty:** Medium · about 15 min · **Practises:** [`groupBy(...).applyInPandas(func, schema)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#groupbyapplyinpandasfunc-schema)

*Per-entity transforms are natural in pandas.*

For each customer, return every order with `rank_in_customer` (1 = largest amount) and `pct_of_max` (amount / the customer's largest amount × 100, rounded to 1), computed in pandas. Columns: `order_id`, `rank_in_customer`, `pct_of_max`, ordered by `order_id`.

In [ ]:
orders.select("order_id", "customer_id", "amount").show()

**Expected output**

| order_id | rank_in_customer | pct_of_max |
| --- | --- | --- |
| 1 | 1 | 100.0 |
| 2 | 2 | 70.4 |
| 3 | 3 | 25.0 |
| 4 | 1 | 100.0 |
| 5 | 2 | 21.2 |
| 6 | 2 | 48.1 |
| 7 | 1 | 100.0 |
| 8 | 2 | 7.6 |
| 9 | 1 | 100.0 |
| 10 | 1 | 100.0 |
| 11 | 1 | 100.0 |
| 12 | 2 | 45.9 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def per_customer(pdf: pd.DataFrame) -> pd.DataFrame:
    amount = pdf["amount"].astype(float)
    pdf["rank_in_customer"] = amount.rank(method="first", ascending=False).astype(int)
    pdf["pct_of_max"] = (amount / amount.max() * 100).round(1)
    return pdf[["order_id", "rank_in_customer", "pct_of_max"]]

result = orders.groupBy("customer_id").applyInPandas(per_customer, "order_id int, rank_in_customer int, pct_of_max double").orderBy("order_id")
result.show()

## Exercise 7: Filter and reshape batches with mapInPandas

**Difficulty:** Medium · about 10 min · **Practises:** [`mapInPandas(func, schema)` and `mapInArrow`](pyspark-reference-12-python-udfs-and-pandas.ipynb#mapinpandasfunc-schema-and-mapinarrow)

*Batch-wise Python processing that can change the row count.*

With `mapInPandas`, keep order lines with qty >= 2 and return `order_id`, `sku` and `line_value` (qty × unit_price as double), ordered by `order_id`, `sku`.

In [ ]:
order_items.show(20)

**Expected output**

| order_id | sku | line_value |
| --- | --- | --- |
| 1 | SKU-4001 | 25.0 |
| 2 | SKU-4009 | 50.0 |
| 4 | SKU-4004 | 111.0 |
| 9 | SKU-4011 | 1180.0 |
| 10 | SKU-4012 | 74.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def multi_unit(batches):
    for pdf in batches:
        keep = pdf[pdf["qty"] >= 2].copy()
        keep["line_value"] = keep["qty"] * keep["unit_price"].astype(float)
        yield keep[["order_id", "sku", "line_value"]]

result = order_items.mapInPandas(multi_unit, "order_id int, sku string, line_value double").orderBy("order_id", "sku")
result.show()

## Exercise 8: As-of merge per customer with cogroup

**Difficulty:** Hard · about 20 min · **Practises:** [`groupBy(...).cogroup(other.groupBy(...)).applyInPandas(func, schema)`](pyspark-reference-12-python-udfs-and-pandas.ipynb#groupbycogroupothergroupbyapplyinpandasfunc-schema)

*pandas.merge_asof per key is a practical as-of join for moderate data sizes.*

For customers 1 and 3, attach to each order the latest `tier` from `tiers` at or before `order_ts` (null if none) using `cogroup` and `merge_asof`. Return `order_id` and `tier`, ordered by `order_id`.

In [ ]:
tiers = spark.createDataFrame([(1, datetime(2026, 1, 5, 10, 0), "silver"), (1, datetime(2026, 1, 5, 11, 0), "gold"), (3, datetime(2026, 1, 8, 0, 0), "bronze")],
                              "customer_id int, ts timestamp, tier string")
tiers.show()

**Expected output**

| order_id | tier |
| --- | --- |
| 1 | null |
| 2 | silver |
| 3 | gold |
| 6 | null |
| 7 | bronze |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def attach(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    if left.empty:
        return pd.DataFrame({"order_id": pd.Series(dtype="int32"), "tier": pd.Series(dtype="object")})
    merged = pd.merge_asof(left.sort_values("order_ts"), right.sort_values("ts")[["ts", "tier"]], left_on="order_ts", right_on="ts", direction="backward")
    return merged[["order_id", "tier"]]

result = (orders.where("customer_id IN (1, 3)").groupBy("customer_id")
          .cogroup(tiers.groupBy("customer_id")).applyInPandas(attach, "order_id int, tier string").orderBy("order_id"))
result.show()

## Exercise 9: Split text into rows with a Python UDTF

**Difficulty:** Medium · about 15 min · **Practises:** [`@F.udtf(returnType=...)`: table functions](pyspark-reference-12-python-udfs-and-pandas.ipynb#fudtfreturntype-table-functions)

*Table functions expand one input into many rows with custom logic.*

Write a UDTF `Words` yielding `(word, length)` for each lower-cased word of its input, register it, and use it with `LATERAL` in SQL over `notes`. Return `id`, `word`, `length`, ordered by `id`, `word`.

In [ ]:
notes = spark.createDataFrame([(1, "Late Delivery"), (2, "Great service")], "id int, text string")
notes.show()

**Expected output**

| id | word | length |
| --- | --- | --- |
| 1 | delivery | 8 |
| 1 | late | 4 |
| 2 | great | 5 |
| 2 | service | 7 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
class WordSplitter:
    def eval(self, text: str):
        for word in (text or "").lower().split():
            yield word, len(word)

Words = F.udtf(WordSplitter, returnType="word string, length int")
spark.udtf.register("words", Words)
result = spark.sql("SELECT n.id, w.word, w.length FROM {n} n, LATERAL words(n.text) w ORDER BY n.id, w.word", n=notes)
result.show()

## Exercise 10: Replace a UDF with built-in functions

**Difficulty:** Medium · about 10 min · **Practises:** [Cost and alternatives](pyspark-reference-12-python-udfs-and-pandas.ipynb#cost-and-alternatives)

*Interviewers and reviewers expect you to avoid UDFs when built-ins exist.*

`slow_clean` lower-cases, trims and collapses inner whitespace in `text`. Rewrite it with built-in functions and return `id`, `clean` and `same` (whether it equals the UDF's output), ordered by `id`.

In [ ]:
messy = spark.createDataFrame([(1, "  Hello   World "), (2, "DATA\tEngineering"), (3, None)], "id int, text string")
slow_clean = F.udf(lambda s: None if s is None else " ".join(s.lower().split()), "string")
messy.show(truncate=False)

**Expected output**

| id | clean | same |
| --- | --- | --- |
| 1 | hello world | true |
| 2 | data engineering | true |
| 3 | null | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
clean = F.regexp_replace(F.lower(F.trim("text")), r"\s+", " ")
result = messy.select("id", clean.alias("clean"), clean.eqNullSafe(slow_clean("text")).alias("same")).orderBy("id")
result.show()

## Exercise 11: Find Python evaluation in a plan

**Difficulty:** Easy · about 5 min · **Practises:** [Arrow-optimized Python UDFs](pyspark-reference-12-python-udfs-and-pandas.ipynb#arrow-optimized-python-udfs)

*Plan reading tells you whether Python runs and how data is transferred.*

For a classic UDF, an Arrow UDF and a pandas UDF applied to `name`, return `kind` and `operator` (the `...EvalPython` operator name found in the plan).

In [ ]:
classic = F.udf(lambda s: s, "string")
arrow = F.udf(lambda s: s, "string", useArrow=True)

@F.pandas_udf("string")
def vectorized(s: pd.Series) -> pd.Series:
    return s

**Expected output**

| kind | operator |
| --- | --- |
| classic | BatchEvalPython |
| arrow | ArrowEvalPython |
| pandas | ArrowEvalPython |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def operator(udf):
    text = "\n".join(plan_lines(customers.select(udf("name"))))
    return re.search(r"(\w*EvalPython)", text).group(1)

result = spark.createDataFrame([("classic", operator(classic)), ("arrow", operator(arrow)), ("pandas", operator(vectorized))], "kind string, operator string")
result.show()

## Exercise 12: Round-trip through pandas with Arrow

**Difficulty:** Easy · about 5 min · **Practises:** [`toPandas`, `createDataFrame(pandas_df)` and Arrow](pyspark-reference-12-python-udfs-and-pandas.ipynb#topandas-createdataframepandas_df-and-arrow)

*Small results often go to pandas for plotting or reporting and back.*

With Arrow enabled, convert revenue per status to pandas, add a `share` column (revenue / total, rounded to 3) in pandas, and convert back. Return `status`, `share`, ordered by `status`.

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

**Expected output**

| status | share |
| --- | --- |
| paid | 0.287 |
| placed | 0.1 |
| returned | 0.045 |
| shipped | 0.568 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
pdf = orders.groupBy("status").agg(F.sum("amount").cast("double").alias("revenue")).toPandas()
pdf["share"] = (pdf["revenue"] / pdf["revenue"].sum()).round(3)
result = spark.createDataFrame(pdf[["status", "share"]].astype({"status": object})).orderBy("status")
result.show()

**Next:** look anything up in [PySpark reference 12: Python UDFs, pandas and Arrow](pyspark-reference-12-python-udfs-and-pandas.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).